<a href="https://colab.research.google.com/github/Ozk18532/INTELIGENCIA-COMPUTACIONAL-Oscar-Mercado/blob/main/9_vectorstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector stores and semantic search



In [1]:
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [2]:
#Clases base
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document

#Implementación de VectorStore
class VectorStore:

    def __init__(self, embedding_model: SentenceTransformer):

        self.embedding_model = embedding_model

        self.documents = []

        self.embeddings = None


    def add_documents(self, documents: list[Document]):

        self.documents.extend(documents)

        texts = [doc.text for doc in self.documents]

        self.embeddings = self.embedding_model.encode(texts)

        print(f"{V}Documentos agregados correctamente{E}")
        print(f"{A}Total de documentos:{E}", len(self.documents))


    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:

        # Embedding del query
        query_embedding = self.embedding_model.encode([query])

        # Similaridad coseno
        similarities = cosine_similarity(
            query_embedding,
            self.embeddings
        )[0]

        # Índices ordenados
        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []

        for idx in top_indices:

            result = SearchResult(
                score=float(similarities[idx]),
                document=self.documents[idx]
            )

            results.append(result)

        return results

## Part II: Filtering by metadata

In [3]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass



---



---



Instalación de librerías

In [4]:
!pip install sentence-transformers scikit-learn pandas numpy -q

In [5]:
# Imports y configuración
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer#⚠️
from sklearn.metrics.pairwise import cosine_similarity

#Colores
V = '\033[92m'  # Verde
R = '\033[91m'  # Rojo
A = '\033[94m'  # Azul
M = '\033[95m'  # Morado
E = '\033[0m'   # Fin color

print(f"{V}Librerías cargadas correctamente{E}")

Librerías cargadas correctamente




---

**Implementación de VectorStore**

En esta actividad se implementará un sistema básico de búsqueda semántica utilizando embeddings y cosine similarity.

Un VectorStore es una estructura que almacena documentos junto con sus representaciones vectoriales (embeddings). Estos embeddings permiten encontrar documentos similares semánticamente incluso cuando las palabras exactas no coinciden.

Para esta práctica se utilizará el dataset Animal Fun Facts Dataset.

Cada documento tendrá:

- Texto principal (`text`)
- Metadatos:
  - `animal_name`
  - `source`
  - `media_link`
  - `wikipedia_link`

Posteriormente se implementará:

- Clase `Document`
- Clase `SearchResult`
- Clase `VectorStore`

Finalmente se realizarán búsquedas semánticas mostrando:
- score de similitud
- texto
- metadatos



---
**Carga del dataset Animal Fun Facts**


In [6]:
import pandas as pd

url = "https://raw.githubusercontent.com/ekohrt/animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"

df = pd.read_csv(url)

print(f"{V}Dataset cargado correctamente{E}")
print(df.shape)

Dataset cargado correctamente
(7734, 5)


In [7]:
print(f"{A}Dimensiones del dataset:{E}", df.shape)

df.head()

Dimensiones del dataset: (7734, 5)


,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark




---
**Conversión del dataset a documentos**






In [8]:
documents = []

for _, row in df.iterrows():

    metadata = {
        "animal_name": str(row["animal_name"]),
        "source": str(row["source"]),
        "media_link": str(row["media_link"]),
        "wikipedia_link": str(row["wikipedia_link"])
    }

    document = Document(
        text=str(row["text"]),
        metadata=metadata
    )

    documents.append(document)

print(f"{V}Documentos creados correctamente{E}")

print(f"{A}Cantidad total:{E}", len(documents))

Documentos creados correctamente
Cantidad total: 7734




---
**Modelo de embeddings**


In [9]:
embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

print(f"{V}Modelo de embeddings cargado correctamente{E}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo de embeddings cargado correctamente



---
**Crear VectorStore y agregar documentos**



In [10]:
vector_store = VectorStore(
    embedding_model=embedding_model
)

vector_store.add_documents(documents)

Documentos agregados correctamente
Total de documentos: 7734




---
**Función para mostrar resultados**


In [11]:
def mostrar_resultados(results):

    for i, result in enumerate(results):

        print(f"{M}=============================={E}")
        print(f"{V}Resultado #{i+1}{E}")

        print(f"{A}Score:{E} {result.score:.4f}")

        print(f"{A}Texto:{E}")
        print(result.document.text)

        print(f"{A}Metadatos:{E}")

        for key, value in result.document.metadata.items():
            print(f"  - {key}: {value}")

        print()



---
**Búsqueda 1**


In [12]:
query = "animals that can fly"

print(f"{R}Consulta:{E}", query)

results = vector_store.search(query)

mostrar_resultados(results)

Consulta: animals that can fly
Resultado #1
Score: 0.7076
Texto:
They don’t fly, they glide.
The only mammal which can independently fly is the bat. Instead, colugas glide which works in the same way as a wingsuit.
Metadatos:
  - animal_name: colugo (flying lemur)
  - source: https://factanimal.com/colugo/
  - media_link: nan
  - wikipedia_link: /wiki/Colugo

Resultado #2
Score: 0.6901
Texto:
Not all birds are able to fly!
Metadatos:
  - animal_name: bird
  - source: https://a-z-animals.com/animals/bird/
  - media_link: nan
  - wikipedia_link: /wiki/Bird

Resultado #3
Score: 0.6481
Texto:
They rarely fly..
They move around on foot most of the time, only taking to the air to reach their nests or for courtship displays.
Metadatos:
  - animal_name: secretary bird
  - source: https://factanimal.com/secretarybird/
  - media_link: nan
  - wikipedia_link: /wiki/Secretarybird

Resultado #4
Score: 0.6437
Texto:
Bats are the only mammals with wings, and the only ones that can truly fly
Metadatos



---
**Búsqueda 2**


In [13]:
query = "animals that live underwater"

print(f"{R}Consulta:{E}", query)#⚠️

results = vector_store.search(query)

mostrar_resultados(results)

Consulta: animals that live underwater
Resultado #1
Score: 0.7298
Texto:
They are semi-aquatic mammals..
They live in and around lakes, rivers, swamps, and tropical rivers. They are excellent at swimming and can hold their breath for 5 minutes at a time. They are well adapted for this, with partially webbed toes for swimming. Also, their eyes, ears and nose are high on their heads to watch out for predators when they are underwater. 2
Metadatos:
  - animal_name: capybara
  - source: https://factanimal.com/capybara/
  - media_link: nan
  - wikipedia_link: /wiki/Capybara

Resultado #2
Score: 0.7203
Texto:
They can stay submerged underwater for 6-8 mins.
They have large lungs, which allows this semi-aquatic animal to stay underwater for extended lengths of time.
Metadatos:
  - animal_name: beaver
  - source: https://factanimal.com/beaver/
  - media_link: nan
  - wikipedia_link: /wiki/Beaver

Resultado #3
Score: 0.6918
Texto:
They can’t survive underwater.
They do still have gills, but the



---
**Búsqueda 3**


In [14]:
query = "dangerous predators"

print(f"{R}Consulta:{E}", query)#⚠️

results = vector_store.search(query)

mostrar_resultados(results)

Consulta: dangerous predators
Resultado #1
Score: 0.6970
Texto:
Has no real natural predators!
Metadatos:
  - animal_name: buffalo
  - source: https://a-z-animals.com/animals/buffalo/
  - media_link: nan
  - wikipedia_link: /wiki/Bison

Resultado #2
Score: 0.6970
Texto:
Has no real natural predators!
Metadatos:
  - animal_name: mountain lion
  - source: https://a-z-animals.com/animals/mountain-lion/
  - media_link: nan
  - wikipedia_link: /wiki/Cougar

Resultado #3
Score: 0.6654
Texto:
A dominant predator in it's environment!
Metadatos:
  - animal_name: brown bear
  - source: https://a-z-animals.com/animals/brown-bear/
  - media_link: nan
  - wikipedia_link: /wiki/Brown_bear

Resultado #4
Score: 0.6632
Texto:
A very bold and ferocious predator!
Metadatos:
  - animal_name: ermine
  - source: https://a-z-animals.com/animals/ermine/
  - media_link: nan
  - wikipedia_link: /wiki/Stoat

Resultado #5
Score: 0.6511
Texto:
In the event of a predator, they will act as a group and fight to prote



---
**Búsqueda 4**



In [15]:
query = "animals with strong memory"

print(f"{R}Consulta:{E}", query)#⚠️

results = vector_store.search(query)

mostrar_resultados(results)

Consulta: animals with strong memory
Resultado #1
Score: 0.5720
Texto:
opossums have better memories than rats, cats, dogs and even pigs.
Metadatos:
  - animal_name: opossum
  - source: /r/AskReddit/comments/gbh7zz/what_are_some_really_amazing_animal_facts/fp65hrs/
  - media_link: nan
  - wikipedia_link: /wiki/Opossum

Resultado #2
Score: 0.5640
Texto:
Some dogs can recognize over 1000 words
Metadatos:
  - animal_name: dog
  - source: https://www.animalfactsencyclopedia.com/All-About-Dogs.html
  - media_link: nan
  - wikipedia_link: /wiki/Dog

Resultado #3
Score: 0.5558
Texto:
The red squirrel has one of the most impressive memories in the entire animal kingdom
Metadatos:
  - animal_name: red squirrel
  - source: https://a-z-animals.com/animals/red-squirrel/
  - media_link: nan
  - wikipedia_link: /wiki/Red_squirrel

Resultado #4
Score: 0.5049
Texto:
Highly active and intelligent dogs!
Metadatos:
  - animal_name: bedlington terrier
  - source: https://a-z-animals.com/animals/bedlington



---
**Búsqueda 5**


In [16]:
query = "fast animals"

print(f"{R}Consulta:{E}", query)#⚠️

results = vector_store.search(query)

mostrar_resultados(results)

Consulta: fast animals
Resultado #1
Score: 0.8064
Texto:
Fastest animal on Earth
Metadatos:
  - animal_name: peregrine falcon
  - source: https://a-z-animals.com/animals/peregrine-falcon/
  - media_link: nan
  - wikipedia_link: /wiki/Peregrine_falcon

Resultado #2
Score: 0.7487
Texto:
The fastest creatures on the planet!
Metadatos:
  - animal_name: falcon
  - source: https://a-z-animals.com/animals/falcon/
  - media_link: nan
  - wikipedia_link: /wiki/Falcon

Resultado #3
Score: 0.6795
Texto:
The fastest land mammal in the world!
Metadatos:
  - animal_name: cheetah
  - source: https://a-z-animals.com/animals/cheetah/
  - media_link: nan
  - wikipedia_link: /wiki/Cheetah

Resultado #4
Score: 0.6764
Texto:
Patagonian mara are the worlds fastest rodent
.
On that topic, they can really move! Plains animals often use speed to their advantage (think cheetahs and gazelles), but the mara is a record-holder in that department.
Metadatos:
  - animal_name: patagonian mara
  - source: https://fact

# Conclusión parcial

Durante esta primera parte se implementó un VectorStore básico utilizando embeddings y cosine similarity.

El sistema logró encontrar documentos similares semánticamente utilizando lenguaje natural, sin depender únicamente de coincidencias exactas de palabras.

Además, se cargó correctamente el Animal Fun Facts Dataset como objetos de tipo `Document`, separando el texto principal de sus metadatos.

Las búsquedas realizadas demostraron cómo los embeddings permiten recuperar información relacionada conceptualmente con cada consulta.



---

# **Explicación de FilteredVectorStore**


En esta segunda parte se implementará un sistema de búsqueda semántica con filtros de metadatos.

La diferencia principal entre `VectorStore` y `FilteredVectorStore` es que este último permite restringir los resultados utilizando información adicional almacenada en los metadatos.

Por ejemplo:
- categoría
- autor
- año
- sentimiento
- tipo de contenido

Esto permite realizar búsquedas mucho más específicas y organizadas.

Para esta práctica se utilizará un pequeño dataset de noticias tecnológicas y deportivas creado manualmente.



---
**Carga del dataset secundario (NIFTY)**

En esta sección se utilizará el dataset **NIFTY**, el cual contiene noticias y registros relacionados con el entorno financiero.  
La información incluye texto natural acompañado de múltiples metadatos, lo que permite realizar búsquedas, filtrados y análisis más completos dentro del VectorStore.

###  Contenido principal
- Titulares de noticias financieras
- Texto descriptivo relacionado con mercados y economía
- Contexto económico asociado a cada registro

###  Metadatos disponibles
- Índices bursátiles
- Fechas y timestamps
- Rankings o puntuaciones
- Labels y categorías financieras

###  Aplicaciones posibles
- Clasificación de texto financiero
- Análisis de lenguaje natural (NLP)
- Búsquedas filtradas por metadata
- Modelado temporal de noticias
- Sistemas de recomendación y recuperación semántica


In [17]:
!pip install datasets -q

from datasets import load_dataset
#⚠️
ds = load_dataset("raeidsaqur/NIFTY")

print(f"{V}Dataset NIFTY cargado correctamente{E}")

print(ds)


Dataset NIFTY cargado correctamente
DatasetDict({
    train: Dataset({
        features: ['id', 'date', 'context', 'news', 'conversations', 'label', 'pct_change'],
        num_rows: 1477
    })
    test: Dataset({
        features: ['id', 'date', 'context', 'news', 'conversations', 'label', 'pct_change'],
        num_rows: 317
    })
    valid: Dataset({
        features: ['id', 'date', 'context', 'news', 'conversations', 'label', 'pct_change'],
        num_rows: 317
    })
})




---
**Exploración del dataset**


In [18]:
df_nifty = ds["train"].to_pandas()

print(f"{A}Dimensiones:{E}", df_nifty.shape)

df_nifty.head()

Dimensiones: (1477, 7)


,id,date,context,news,conversations,label,pct_change
0,nifty_0,2010-01-06,"date,open,high,low,close,adj_close,volume,pct_...",China Officials Likely Knew of Bad Milk\nSony'...,"[{'role': 'user', 'value': 'Project the $SPY i...",Neutral,0.0042
1,nifty_1,2010-01-07,"date,open,high,low,close,adj_close,volume,pct_...",Britain Set for Next Step on Wind Power\nJourn...,"[{'role': 'user', 'value': 'Analyze market dat...",Neutral,0.0033
2,nifty_2,2010-01-11,"date,open,high,low,close,adj_close,volume,pct_...","HK, China stocks rise on reforms; brokerages s...","[{'role': 'user', 'value': 'Forecast the $SPY ...",Fall,-0.0093
3,nifty_3,2010-01-12,"date,open,high,low,close,adj_close,volume,pct_...",DemandTec Names Retail Challenge Grand Final W...,"[{'role': 'user', 'value': 'Forecast the $SPY ...",Rise,0.0084
4,nifty_4,2010-01-13,"date,open,high,low,close,adj_close,volume,pct_...",Gear Malfunction Suspected in United Jet Incid...,"[{'role': 'user', 'value': 'Forecast the $SPY ...",Neutral,0.0027




---
**Ver columnas disponibles**


In [19]:
print(f"{V}Columnas disponibles:{E}")

print(df_nifty.columns)#⚠️

Columnas disponibles:
Index(['id', 'date', 'context', 'news', 'conversations', 'label',
       'pct_change'],
      dtype='object')




---
**Crear FilteredVectorStore**

Dataset secundario: NIFTY

Para esta segunda parte se utilizará el dataset NIFTY.

Este dataset contiene textos relacionados con noticias financieras y económicas acompañadas de diferentes metadatos.

El objetivo será implementar un `FilteredVectorStore` que permita realizar búsquedas semánticas restringidas utilizando filtros sobre dichos metadatos.




---
**Conversión del dataset NIFTY a documentos**


In [20]:
filtered_documents = []

for _, row in df_nifty.iterrows():

    metadata = {

        "id": str(row["id"]),

        "date": str(row["date"]),

        "context": str(row["context"]),

        "label": str(row["label"]),

        "pct_change": str(row["pct_change"])
    }

    document = Document(

        text=str(row["news"]),

        metadata=metadata
    )

    filtered_documents.append(document)

print(f"{V}Documentos creados correctamente{E}")

print(f"{A}Cantidad total:{E}", len(filtered_documents))

Documentos creados correctamente
Cantidad total: 1477




---
**Visualización de ejemp**lo


In [21]:
example_doc = filtered_documents[0]

print(f"{M}=============================={E}")

print(f"{V}Texto:{E}")

print(example_doc.text)

print()

print(f"{A}Metadatos:{E}")

for key, value in example_doc.metadata.items():#⚠️

    print(f"- {key}: {value}")

Texto:
China Officials Likely Knew of Bad Milk
Sony's CEO on Strategy in 3-D Technology
Copper Settles at 16-Month High
Gold Ends Near 3-Week High
Kraft Gets Antitrust Clearance
European Stocks Close Flat
M&S Sales Miss Expectations
Yen Boosts Japan's Exporters
Future Group: Value Retail Unit May Consider IPO
Dollar, Yen Gain on Haven Plays
Clothing Sales Sagged in December
Luxury Logos Draw Asian Shoppers
Mexican Stocks Have Record in Sight
Cold Snap Heats Up OJ Prices
Citi's Havens Got $9 Million for '09
TCW Will Quit U.S. Program
Scripps to Offer Free Show in Cablevision Feud
Two Pilots Die in Small-Jet Crash

Metadatos:
- id: nifty_0
- date: 2010-01-06 00:00:00
- context: date,open,high,low,close,adj_close,volume,pct_change,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma
- label: Neutral
- pct_change: 0.0042




---
**Implementación de FilteredVectorStore**


In [22]:
class FilteredVectorStore:

    def __init__(self, embedding_model: SentenceTransformer):#⚠️

        self.embedding_model = embedding_model

        self.documents = []

        self.embeddings = None


    def add_documents(self, documents: list[Document]):

        self.documents.extend(documents)

        texts = [doc.text for doc in self.documents]

        self.embeddings = self.embedding_model.encode(texts)

        print(f"{V}Documentos agregados correctamente{E}")

        print(f"{A}Total de documentos:{E}", len(self.documents))


    def search(
        self,
        query: str,
        top_k: int = 2,
        metadata_filter: dict[str, str] | None = None
    ) -> list[SearchResult]:

        filtered_docs = []

        filtered_embeddings = []

        for idx, doc in enumerate(self.documents):

            include_document = True

            if metadata_filter is not None:

                for key, value in metadata_filter.items():

                    if doc.metadata.get(key) != value:

                        include_document = False

                        break

            if include_document:#⚠️

                filtered_docs.append(doc)

                filtered_embeddings.append(self.embeddings[idx])

        if len(filtered_docs) == 0:

            print(f"{R}No se encontraron documentos con ese filtro{E}")

            return []

        query_embedding = self.embedding_model.encode([query])

        similarities = cosine_similarity(
            query_embedding,
            filtered_embeddings
        )[0]

        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []

        for idx in top_indices:

            result = SearchResult(

                score=float(similarities[idx]),

                document=filtered_docs[idx]
            )

            results.append(result)

        return results



---
Crear FilteredVectorStore


In [23]:
filtered_vector_store = FilteredVectorStore(
    embedding_model=embedding_model#⚠️
)

filtered_vector_store.add_documents(filtered_documents)

Documentos agregados correctamente
Total de documentos: 1477


In [24]:
print(f"{V}Valores únicos de label:{E}")

print(df_nifty["label"].unique())

print()

print(f"{V}Valores únicos de context:{E}")

print(df_nifty["context"].unique()[:10])

print()

print(f"{V}Primeras fechas:{E}")

print(df_nifty["date"].head())

Valores únicos de label:
['Neutral' 'Fall' 'Rise']

Valores únicos de context:
['date,open,high,low,close,adj_close,volume,pct_change,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma']

Primeras fechas:
0   2010-01-06
1   2010-01-07
2   2010-01-11
3   2010-01-12
4   2010-01-13
Name: date, dtype: datetime64[s]




---
**Búsqueda filtrada 1**


In [25]:
query = "stock market growth"

metadata_filter = {
    "label": "Rise"
}

print(f"{R}Consulta:{E}", query)

print(f"{A}Filtro:{E}", metadata_filter)

results = filtered_vector_store.search(#⚠️
    query=query,
    metadata_filter=metadata_filter
)

mostrar_resultados(results)

Consulta: stock market growth
Filtro: {'label': 'Rise'}
Resultado #1
Score: 0.4521
Texto:
Market Chatter- Corporate finance press digest
INDIA PRESS-Oberois to invest 10 bln rupees in Bangalore, Goa-Times of India
CNH Tracker-Banks launch new yuan FX options for companies in China
It may be too early to give up on the bull market in equities
MOVES-Vontobel Financial Products Asia Pacific names Thomas Sussli as head
PRESS DIGEST - Wall Street Journal - Aug 7
PRESS DIGEST- New York Times business news - Aug 7
China shares fall on weak energy, financials; HK down on casinos
Fraport Interim Report - 6 Months 2014: Financial Figures Grow as Expected
Japan's GPIF to boost stock allocation to over 20 pct - sources
Elektrobit Corporation's (EB) Interim Report January-June 2014
African Markets - Factors to watch on Aug 7
Deutsche Telekom Q2 core profit beats view on growth in U.S.
BRIEF-Inficon Holding says Q2 2014 sales of USD 74.8 million
NestlÃ© S.A. : First Half 2014: 4.7% organic growth in



---
**Búsqueda filtrada 2**


In [26]:
query = "financial losses"

metadata_filter = {
    "label": "Fall"
}

print(f"{R}Consulta:{E}", query)#⚠️

print(f"{A}Filtro:{E}", metadata_filter)

results = filtered_vector_store.search(
    query=query,
    metadata_filter=metadata_filter
)

mostrar_resultados(results)

Consulta: financial losses
Filtro: {'label': 'Fall'}
Resultado #1
Score: 0.4020
Texto:
Two Firms Enter Clean-Air Settlement
Tullow's CEO to Meet with Uganda's President
Aldi Faces Suit From Store Managers
Foreign Buyers Snap Up Japanese Shares
Bloomies to Open Outlet Stores
BSkyB Loses Appeal Over ITV Stake
Weaker Yen Lifts Japan Shares
U.S. Sales Boost Ahold's Revenue
Fifth Third Bancorp's Loss Shrinks
Indian Shares End 2.4% Lower
N.Y. Times to Charge for Web
AXA Unit Raises Profit Estimate
Falling Beer Sales Pressure Brewers
Paris Urges Areva, EDF to End Spat
Cockpit-Door Defects Worry Officials
Dubai World Loses Top Executive
Global Jitters Boost Dollar
LaBranche Realigns After Specialist Sale
Icahn Wins Bid for Fontainebleau
GenRe Reaches Deal in AIG Case
Metadatos:
  - id: nifty_8
  - date: 2010-01-21 00:00:00
  - context: date,open,high,low,close,adj_close,volume,pct_change,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma
  - label: Fall
  - pct_change: -0.0223




---
**Búsqueda filtrada 3**


In [27]:
query = "stable market"

metadata_filter = {
    "label": "Neutral"
}

print(f"{R}Consulta:{E}", query)#⚠️

print(f"{A}Filtro:{E}", metadata_filter)

results = filtered_vector_store.search(
    query=query,
    metadata_filter=metadata_filter
)

mostrar_resultados(results)

Consulta: stable market
Filtro: {'label': 'Neutral'}
Resultado #1
Score: 0.4373
Texto:
This Holiday Season, Shoppers Find It More Invasive to be Targeted In-Store than Online
Consider Canada City Alliance Representing 59% of Canadian GDP Delivers Infrastructure Opportunity Message on CETA During Three-City European Investment Mission
ECB's Noyer: financial markets stabilising, rates returning to normal
ECB's Noyer: financial markets stabilizing, rates returning to normal
Sasol Chief Financial Officer Update
Australia shares edge higher on Wall St, banks, weak $A underpin
FOREX -Yen skids to 6-month low vs dollar, 4-year low vs euro
UPDATE 1-China Everbright Bank says profit, NPLs up ahead of HK listing
BOJ's Kuroda: Impact of negative rates on economy unknown
BOJ's Kuroda: Impact of negative rates on economy unknown
PRESS DIGEST- New York Times business news - Nov 25
PRESS DIGEST - Wall Street Journal - Nov 25
Polarcus Financial Calendar 2014
China reaffirms ban on housing sales on rur



---
**Búsqueda filtrada 4**


In [28]:
query = "company profits"

metadata_filter = {
    "label": "Rise"
}

print(f"{R}Consulta:{E}", query)#⚠️

print(f"{A}Filtro:{E}", metadata_filter)

results = filtered_vector_store.search(
    query=query,
    metadata_filter=metadata_filter
)

mostrar_resultados(results)

Consulta: company profits
Filtro: {'label': 'Rise'}
Resultado #1
Score: 0.4019
Texto:
After Financial Crisis, Goldman  Goes It Alone
Tesla’s Loss Widens as Deliveries Fall Short
Waldorf Astoria Hotel Sale Completed
Euro Turns Higher on Reports of Greece Agreement
Whole Foods Profit Rises 5.7%
Pilgrim’s Pride Profit Up 17%
W.R. Grace: The End of an Empire
An Opportunity Opens for NBC Rivals
Corporate Watch: News Digest
Sysco, FTC Meet About US Foods Deal
Cheesecake Factory Profit Falls on Higher Costs
USA Today Pages to Appear in Other Papers
SEC Probes CVR Disclosure During Takeover
Hedge Funds Get Big Payoff on Currencies
Sun Life Financial Misses Analysts’ Expectations
Edict by Beijing Spells Trouble for Bonds
GE to Allow Proxy Access for Big Investors
Former Petrobras CEO Weighs In on Corruption Probe
Panera Bread’s Profit Falls 11%
Zulily Sees First-Quarter Sales Below Analyst Views
Kraft’s Shares Look a Bit Overcooked
Mondelez Profit Hit By Currency Effects, Higher Costs
MetLife’s



---
**Búsqueda filtrada 5**


In [29]:
query = "economic crisis"

metadata_filter = {
    "label": "Fall"
}

print(f"{R}Consulta:{E}", query)

print(f"{A}Filtro:{E}", metadata_filter)

results = filtered_vector_store.search(
    query=query,
    metadata_filter=metadata_filter
)

mostrar_resultados(results)

Consulta: economic crisis
Filtro: {'label': 'Fall'}
Resultado #1
Score: 0.4059
Texto:
Mylan Set to Lose in $26 Billion Hostile Takeover Battle for Perrigo
India Shares Fall Along With Rest of the Region
Deutsche Bank Names New Head of Corporate, Investment Bank in Asia Pacific
Ethan Allen’s Chief in Fight of His Life
Copper Swoon Presses Glencore, Other Miners
Stock Decline Picks Up Steam
Nordstrom Earnings Fall Sharply
Fed Weighs Tightening Revolving-Door Curbs
LoanDepot Postpones IPO
Fischer: Strong Dollar Delayed Rate Rise, but Fed Could Move in December
Hottest Energy Trade: A Ride Aboard the Colonial Pipeline
China’s Stock Crackdown: ‘Kill the Chicken to Scare the Monkey’
Activist Investors Seek Changes to ITG Board
Warren Buffett Has an Image Problem
Petrobras Reports Third-Quarter Loss
Peru’s Central Bank Holds Interest Rate Steady
Calpers Sells $3 Billion of Real-Estate Holdings to Blackstone Unit
Brazil’s Inflation Struggles Spark Concerns About ‘Fiscal Dominance’
Yum Brands R

# Conclusión parcial

En esta segunda parte se logró implementar correctamente un FilteredVectorStore, permitiendo realizar búsquedas semánticas utilizando filtros de metadatos.

A diferencia del VectorStore básico, esta versión permite recuperar información más específica dependiendo de atributos como fecha, contexto, etiqueta o cambios porcentuales, lo que ayuda a organizar mejor los resultados obtenidos.

Aunque apenas estamos comenzando a trabajar con este tipo de herramientas, la actividad permitió entender de forma más clara cómo funcionan los sistemas modernos de recuperación de información basados en embeddings y filtros estructurados.

Además, se pudo observar cómo la combinación entre búsqueda semántica y metadatos mejora la precisión de las consultas, acercándonos al funcionamiento real de muchas aplicaciones actuales de inteligencia artificial y procesamiento de lenguaje natural.



---



---



---



⚠️ Disclaimer Algunas partes de este notebook fueron asistidas mediante herramientas de inteligencia artificial.

Estas secciones han sido revisadas, comprendidas y adaptadas para cumplir con los objetivos de la actividad.

Las partes asistidas están claramente identificadas con el siguiente icono: #⚠️